# 04. Decomposed SIR Evaluation (Symbolic)

**Purpose:** Compute the decomposed Semantic Irrelevance Ratio (SIR*) over the full KG path set and the symbolically filtered path set.

**What SIR measures (§3.4):** At each decoding step $t$, what fraction of paths in the valid token set are semantically irrelevant? Irrelevance is the disjunction of two independent failure modes:
- **Type irrelevance** ($\text{SIR}^*_{\text{type}}$): terminal entity type mismatches the question's answer type
- **Trajectory irrelevance** ($\text{SIR}^*_{\text{traj}}$): a relation in the path has a declared range incompatible with the question's answer type

**Reference:** Chapter 3, §3.4. Equations (3.5)-(3.11).

Key formulas:
```
irrel_type(p, q) = 1[type(e_h) ∉ T(q, h)]                          (Eq. 3.5)
irrel_traj(p, q) = 1[∃ hop i: range(r_i) ∩ T(q, h) = ∅]           (Eq. 3.6)
irrel*(p, q) = irrel_type ∨ irrel_traj                              (Eq. 3.7)
SIR*_type(q,t) = count(irrel_type=1) / |P(q,t)|                    (Eq. 3.8)
SIR*_traj(q,t) = count(irrel_traj=1) / |P(q,t)|                    (Eq. 3.9)
SIR*(q,t) = count(irrel*=1) / |P(q,t)|                             (Eq. 3.10)
```

In [ ]:
import os
import sys
import json
import time
import hashlib

!pip install -q datasets marisa-trie networkx

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print("Not on Colab -- skipping Drive mount")

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca
%cd graph-constrained-reasoning
sys.path.insert(0, '.')
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))

from collections import defaultdict

from tqdm import tqdm
import numpy as np
from datasets import load_dataset
import src.utils as utils
from type_oracle import TypeOracle, compute_type_irrelevance

## Configuration

In [ ]:
# Dataset
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"
INDEX_LEN = 2

# Sampling
MAX_SAMPLES = 100

# Google Drive output
DRIVE_BASE = "/content/drive/MyDrive/graph-constrained-reasoning/results" if ON_COLAB else "./results"
CACHE_DIR = os.path.join(DRIVE_BASE, "sir_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

## Load Dataset

In [ ]:
dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

print(f"Loaded {len(dataset)} questions")
print(f"Columns: {dataset.column_names}")
sample = dataset[0]
print(f"\nSample question: {sample['question']}")
print(f"Sample paths count: {len(sample.get('paths', []))}")

## Symbolic SIR Computation

In [ ]:
def compute_symbolic_sir(paths_list, oracle, answer_types, max_hop):
    """Compute decomposed SIR metrics for a single path set.

    Parameters
    ----------
    paths_list : list of list of (head, relation, tail) tuples
        Candidate paths from the KG.
    oracle : TypeOracle
        Symbolic oracle with type/range gate methods.
    answer_types : frozenset of str
        Inferred answer types from the question.
    max_hop : int
        Maximum path length considered.

    Returns
    -------
    dict with keys:
      sir_type : fraction of paths with type irrelevance (Eq. 3.8)
      sir_traj : fraction of paths with trajectory irrelevance (Eq. 3.9)
      sir : fraction of paths with either (Eq. 3.10)
      hop_sir_type, hop_sir_traj, hop_sir, hop_counts : per-hop breakdowns
    """
    paths_by_hop = defaultdict(list)
    for p in paths_list:
        paths_by_hop[len(p)].append(p)

    hop_sir_type = {}
    hop_sir_traj = {}
    hop_sir = {}
    hop_counts = {}

    for step, hop_paths in paths_by_hop.items():
        n_total = len(hop_paths)
        hop_counts[step] = n_total

        n_type_irr = 0
        n_traj_irr = 0
        n_combined = 0

        for p in hop_paths:
            terminal_entity = p[-1][2]
            h = len(p)

            # Type irrelevance (Eq. 3.5)
            type_bad = not oracle.type_gate(
                terminal_entity, answer_types, h, max_hop
            )

            # Trajectory irrelevance (Eq. 3.6)
            traj_bad = False
            for _, relation, _ in p:
                rel_schema = oracle._schema.get(relation)
                if rel_schema is None:
                    continue
                range_types = rel_schema.get("range", frozenset())
                if not range_types:
                    continue
                if not (range_types & answer_types):
                    traj_bad = True
                    break

            if type_bad:
                n_type_irr += 1
            if traj_bad:
                n_traj_irr += 1
            if type_bad or traj_bad:
                n_combined += 1

        hop_sir_type[step] = n_type_irr / max(1, n_total)
        hop_sir_traj[step] = n_traj_irr / max(1, n_total)
        hop_sir[step] = n_combined / max(1, n_total)

    # Aggregate across hops
    all_type = list(hop_sir_type.values())
    all_traj = list(hop_sir_traj.values())
    all_sir = list(hop_sir.values())

    return {
        "sir_type": np.mean(all_type) if all_type else 0.0,
        "sir_traj": np.mean(all_traj) if all_traj else 0.0,
        "sir": np.mean(all_sir) if all_sir else 0.0,
        "hop_sir_type": hop_sir_type,
        "hop_sir_traj": hop_sir_traj,
        "hop_sir": hop_sir,
        "hop_counts": hop_counts,
    }

## Full KG SIR (GCR Baseline)

In [ ]:
def _cache_key(prefix, idx):
    """Return a filesystem-safe cache key for a given question index."""
    raw = f"{prefix}_{DATASET}_{SPLIT}_{idx}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


def _load_cache(key):
    """Load a cached result from disk, or return None."""
    path = os.path.join(CACHE_DIR, f"{key}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return None


def _save_cache(key, data):
    """Save a result dict to the cache."""
    path = os.path.join(CACHE_DIR, f"{key}.json")
    with open(path, "w") as f:
        json.dump(data, f)


all_sir_type = []
all_sir_traj = []
all_sir = []
hop_sir_type_agg = defaultdict(list)
hop_sir_traj_agg = defaultdict(list)
hop_sir_agg = defaultdict(list)
hop_count_agg = defaultdict(list)

for idx, data in enumerate(tqdm(dataset, desc="Full KG SIR")):
    question = data["question"]
    entities = data["q_entity"] if "q_entity" in data else []

    ck = _cache_key("full", idx)
    cached = _load_cache(ck)
    if cached is not None:
        result = cached
    else:
        try:
            oracle = TypeOracle.from_graph(data["graph"])
            answer_types = oracle.infer_answer_types(question)

            if "paths" in data:
                paths_list = data["paths"]
            else:
                g = utils.build_graph(data["graph"])
                paths_list = utils.dfs(g, entities, INDEX_LEN)

            if len(paths_list) == 0:
                continue

            result = compute_symbolic_sir(
                paths_list, oracle, answer_types, INDEX_LEN
            )
            _save_cache(ck, result)
        except Exception as e:
            print(f"  Error on question {idx}: {e}")
            continue

    for step in result["hop_sir_type"]:
        hop_sir_type_agg[step].append(result["hop_sir_type"][step])
        hop_sir_traj_agg[step].append(result["hop_sir_traj"][step])
        hop_sir_agg[step].append(result["hop_sir"][step])
        hop_count_agg[step].append(result["hop_counts"][step])
        all_sir_type.append(result["hop_sir_type"][step])
        all_sir_traj.append(result["hop_sir_traj"][step])
        all_sir.append(result["hop_sir"][step])

print(f"Corpus-level SIR (GCR baseline, {len(all_sir)} observations):")
print(f"  SIR*_type: {np.mean(all_sir_type):.4f}")
print(f"  SIR*_traj: {np.mean(all_sir_traj):.4f}")
print(f"  SIR*:      {np.mean(all_sir):.4f}")

print("\nPer-hop SIR:")
for step in sorted(hop_sir_agg.keys()):
    n = len(hop_sir_agg[step])
    print(
        f"  Hop-{step}: SIR*={np.mean(hop_sir_agg[step]):.4f}  "
        f"type={np.mean(hop_sir_type_agg[step]):.4f}  "
        f"traj={np.mean(hop_sir_traj_agg[step]):.4f}  "
        f"(n={n})"
    )

## DCA-Trie v1 SIR (Filtered Paths)

In [ ]:
all_sir_type_v1 = []
all_sir_traj_v1 = []
all_sir_v1 = []
hop_sir_type_v1 = defaultdict(list)
hop_sir_traj_v1 = defaultdict(list)
hop_sir_v1 = defaultdict(list)
hop_count_v1 = defaultdict(list)
n_empty_v1 = 0

for idx, data in enumerate(tqdm(dataset, desc="DCA-v1 SIR")):
    question = data["question"]
    entities = data["q_entity"] if "q_entity" in data else []

    ck = _cache_key("dca_v1", idx)
    cached = _load_cache(ck)
    if cached is not None:
        result = cached
        if result is None:
            n_empty_v1 += 1
            continue
    else:
        try:
            oracle = TypeOracle.from_graph(data["graph"])
            answer_types = oracle.infer_answer_types(question)

            if "paths" in data:
                paths_list = data["paths"]
            else:
                g = utils.build_graph(data["graph"])
                paths_list = utils.dfs(g, entities, INDEX_LEN)

            if len(paths_list) == 0:
                _save_cache(ck, None)
                n_empty_v1 += 1
                continue

            # Apply symbolic gates to filter (same as Algorithm 1)
            filtered_paths = []
            for p in paths_list:
                h = len(p)
                admit = True

                for _, relation, tail_entity in p:
                    if not oracle.range_gate(relation, tail_entity):
                        admit = False
                        break

                if admit:
                    terminal_entity = p[-1][2]
                    if not oracle.type_gate(
                        terminal_entity, answer_types, h, INDEX_LEN
                    ):
                        admit = False

                if admit:
                    filtered_paths.append(p)

            if len(filtered_paths) == 0:
                _save_cache(ck, None)
                n_empty_v1 += 1
                continue

            result = compute_symbolic_sir(
                filtered_paths, oracle, answer_types, INDEX_LEN
            )
            _save_cache(ck, result)
        except Exception as e:
            print(f"  Error on question {idx}: {e}")
            continue

    for step in result["hop_sir_type"]:
        hop_sir_type_v1[step].append(result["hop_sir_type"][step])
        hop_sir_traj_v1[step].append(result["hop_sir_traj"][step])
        hop_sir_v1[step].append(result["hop_sir"][step])
        hop_count_v1[step].append(result["hop_counts"][step])
        all_sir_type_v1.append(result["hop_sir_type"][step])
        all_sir_traj_v1.append(result["hop_sir_traj"][step])
        all_sir_v1.append(result["hop_sir"][step])

print(f"Corpus-level SIR (DCA-Trie v1, {len(all_sir_v1)} observations, "
      f"{n_empty_v1} empty):")
print(f"  SIR*_type: {np.mean(all_sir_type_v1):.4f}")
print(f"  SIR*_traj: {np.mean(all_sir_traj_v1):.4f}")
print(f"  SIR*:      {np.mean(all_sir_v1):.4f}")

print("\nPer-hop SIR:")
for step in sorted(hop_sir_v1.keys()):
    n = len(hop_sir_v1[step])
    print(
        f"  Hop-{step}: SIR*={np.mean(hop_sir_v1[step]):.4f}  "
        f"type={np.mean(hop_sir_type_v1[step]):.4f}  "
        f"traj={np.mean(hop_sir_traj_v1[step]):.4f}  "
        f"(n={n})"
    )

## Comparison

In [ ]:
def reduction(gcr, dca):
    """Compute percentage reduction from GCR to DCA values."""
    return (gcr - dca) / max(gcr, 1e-8) * 100


if len(all_sir) > 0 and len(all_sir_v1) > 0:
    print(f"{'Metric':<20} {'GCR':<10} {'DCA-v1':<10} {'Reduction':<10}")
    print("-" * 50)
    print(f"{'SIR*':<20} {np.mean(all_sir):<10.4f} "
          f"{np.mean(all_sir_v1):<10.4f} "
          f"{reduction(np.mean(all_sir), np.mean(all_sir_v1)):<10.1f}%")
    print(f"{'SIR*_type':<20} {np.mean(all_sir_type):<10.4f} "
          f"{np.mean(all_sir_type_v1):<10.4f} "
          f"{reduction(np.mean(all_sir_type), np.mean(all_sir_type_v1)):<10.1f}%")
    print(f"{'SIR*_traj':<20} {np.mean(all_sir_traj):<10.4f} "
          f"{np.mean(all_sir_traj_v1):<10.4f} "
          f"{reduction(np.mean(all_sir_traj), np.mean(all_sir_traj_v1)):<10.1f}%")

    print("\nPer-hop comparison:")
    all_steps = sorted(set(list(hop_sir_agg.keys()) + list(hop_sir_v1.keys())))
    for step in all_steps:
        gcr_s = np.mean(hop_sir_agg.get(step, [0]))
        v1_s = np.mean(hop_sir_v1.get(step, [0]))
        gcr_st = np.mean(hop_sir_type_agg.get(step, [0]))
        v1_st = np.mean(hop_sir_type_v1.get(step, [0]))
        gcr_str = np.mean(hop_sir_traj_agg.get(step, [0]))
        v1_str = np.mean(hop_sir_traj_v1.get(step, [0]))

        print(f"\n  Hop-{step}:")
        print(f"    SIR*:      {gcr_s:.4f} -> {v1_s:.4f} "
              f"({reduction(gcr_s, v1_s):.1f}%)")
        print(f"    SIR*_type: {gcr_st:.4f} -> {v1_st:.4f} "
              f"({reduction(gcr_st, v1_st):.1f}%)")
        print(f"    SIR*_traj: {gcr_str:.4f} -> {v1_str:.4f} "
              f"({reduction(gcr_str, v1_str):.1f}%)")

    # Save summary
    summary = {
        "gcr": {
            "sir": float(np.mean(all_sir)),
            "sir_type": float(np.mean(all_sir_type)),
            "sir_traj": float(np.mean(all_sir_traj)),
        },
        "dca_v1": {
            "sir": float(np.mean(all_sir_v1)),
            "sir_type": float(np.mean(all_sir_type_v1)),
            "sir_traj": float(np.mean(all_sir_traj_v1)),
        },
    }
    summary_path = os.path.join(DRIVE_BASE, "sir_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nSaved summary to {summary_path}")
else:
    print("Insufficient data for comparison")

## Per-Gate Attribution

In [ ]:
print("""SIR*_type  = fraction of paths where terminal entity type != answer type
             -> these are paths the type gate would prune

SIR*_traj  = fraction of paths where a relation's range != answer type
             -> these are paths the range gate would prune

If SIR*_type = 0.6 and SIR*_traj = 0.7, that means:
  - 60% of paths have the wrong terminal type
  - 70% have at least one relation with incompatible range
  - The combined SIR* is NOT 1.3 because these overlap

These two metrics directly measure the two design gaps from
Chapter 2: type blindness (SIR*_type) and trajectory blindness
(SIR*_traj). Each is actionable independently.
""")

# Venn overlap analysis
total_paths = 0
n_type_only = 0
n_traj_only = 0
n_both = 0
n_neither = 0

for idx, data in enumerate(tqdm(dataset, desc="Overlap analysis")):
    entities = data["q_entity"] if "q_entity" in data else []

    try:
        oracle = TypeOracle.from_graph(data["graph"])
        answer_types = oracle.infer_answer_types(data["question"])

        if "paths" in data:
            paths_list = data["paths"]
        else:
            g = utils.build_graph(data["graph"])
            paths_list = utils.dfs(g, entities, INDEX_LEN)

        for p in paths_list:
            h = len(p)
            total_paths += 1

            terminal_entity = p[-1][2]
            type_bad = not oracle.type_gate(
                terminal_entity, answer_types, h, INDEX_LEN
            )

            traj_bad = False
            for _, relation, _ in p:
                rel_schema = oracle._schema.get(relation)
                if rel_schema is None:
                    continue
                range_types = rel_schema.get("range", frozenset())
                if not range_types:
                    continue
                if not (range_types & answer_types):
                    traj_bad = True
                    break

            if type_bad and traj_bad:
                n_both += 1
            elif type_bad:
                n_type_only += 1
            elif traj_bad:
                n_traj_only += 1
            else:
                n_neither += 1
    except Exception as e:
        continue

if total_paths > 0:
    print(f"\nVenn overlap (total paths: {total_paths}):")
    print(f"  Type only:   {n_type_only} ({n_type_only/total_paths*100:.1f}%)")
    print(f"  Traj only:   {n_traj_only} ({n_traj_only/total_paths*100:.1f}%)")
    print(f"  Both:        {n_both} ({n_both/total_paths*100:.1f}%)")
    print(f"  Neither:     {n_neither} ({n_neither/total_paths*100:.1f}%)")

    overlap = n_both / max(1, n_type_only + n_traj_only + n_both)
    print(f"\n  Jaccard overlap of type & traj failures: {overlap:.3f}")
else:
    print("No paths found for overlap analysis")

## Interpretation

In [ ]:
print("""
INTERPRETATION (§3.4.4)

SIR*_type isolates the contribution of type blindness.
  - GCR: high (no type filtering)
  - DCA-Trie v1: ~0 (type gate kills these)

SIR*_traj isolates trajectory blindness.
  - GCR: high (no range filtering)
  - DCA-Trie v1: low (range gate kills these)

The Venn overlap analysis shows whether type and trajectory
irrelevance are independent failure modes or correlate.

Both metrics are purely symbolic -- no embeddings, no thresholds.
They respond directly to Gap 3 in Chapter 2 by attributing
the contribution of each filtering mechanism independently.
""")